In [1]:
import pandas as pd
import numpy as np

In [29]:
df_transaction = pd.read_csv('transaction_cleaned.csv')

In [60]:
df_transfer = pd.read_csv('transfer_cleaned.csv')

In [3]:
df_referral = pd.read_csv('referral_history_cleaned.csv')

## Tốc độ thực hiện hành vi 

| Mức độ |	Điều kiện |	Giải thích | Điểm |
|:-------|:-----------|:-----------|:-----|
|Cao |	Thao tác liên tục < 1s | Gần như chắc chắn dùng script | 3 |
|Trung bình | 1s ≤ thao tác < 5s | Khả năng bot cao |	2 |
|Thấp |	5s ≤ thao tác < 15s | Có thể người nhưng nghi ngờ |	1 |
|Bình thường |	≥ 15s |	Người thật thao tác tự nhiên |	0 |


In [30]:
df_tx = df_transaction.copy()
df_tx['reqDate'] = pd.to_datetime(df_tx['reqDate'])

In [31]:
# Sắp xếp theo user và thời gian
df_tx = df_tx.sort_values(by=['userID', 'reqDate'])

# Tính thời gian cách nhau giữa 2 giao dịch liền kề
df_tx['reqDate'] = df_tx.groupby('userID')['reqDate'].diff().dt.total_seconds()

# Lấy khoảng thời gian NGẮN NHẤT giữa hai giao dịch của mỗi user
min_time_diff_per_user = df_tx.groupby('userID')['reqDate'].min().reset_index()
min_time_diff_per_user.columns = ['userID', 'min_time_diff_seconds']

# Gán điểm theo điều kiện
def calc_score(seconds):
    if pd.isna(seconds):
        return 0
    elif seconds < 1:
        return 3
    elif seconds < 5:
        return 2
    elif seconds < 15:
        return 1
    else:
        return 0

min_time_diff_per_user['score_time_diff'] = min_time_diff_per_user['min_time_diff_seconds'].apply(calc_score)
# Merge điểm trở lại df_transaction
df_transaction = df_transaction.merge(min_time_diff_per_user[['userID', 'score_time_diff']], on='userID', how='left')
df_transaction


,transID,userID,appID,transStatus,deviceID,platform,userChargeAmount,amount,reqDate,campaignID,discountAmount,cashbackTime,score_time_diff
0,2b60a070e297782422257f96ec001e1d,b4cd6bf6334153f048a824fd8c832c06,NaN,1,NaN,NaN,0,50000,2022-03-02 00:00:00,9080,50000,2022-03-02 00:00:00,0
1,83f7f779d15c7d64e30197c472e81219,c69b7376c04ace223a597bb495b6433d,NaN,1,NaN,NaN,0,50000,2022-03-02 00:00:00,9080,50000,2022-03-02 00:00:00,0
2,c3a55d5a75e4eda4a1c9f1817b6f59f5,2a11b577bfde798dce593467ff6c8472,NaN,1,NaN,NaN,0,50000,2022-03-02 00:00:00,9080,50000,2022-03-02 00:00:00,0
3,5660d432965c9081fca164590b86d4e9,898f3699816f0ed166aa5d0c40393cb2,NaN,1,NaN,NaN,0,50000,2022-03-02 00:00:00,9080,50000,2022-03-02 00:00:00,0
4,42bddb3b489c9cf2a795e315a0cf4cbe,d1f55da25d8313b0221f513e62e81cc2,15.0,1,8A3E7FFF-87D0-4513-9F5E-D2F6A5DE60FA,platform1,495000,500000,2022-03-02 00:01:01,8901,5000,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1357393,319c7d437de329fdf7e734dae1fcfbc9,b9bab616c0289035af66278ca7c6651a,12.0,1,D189A8C4AB98337A7071E9FD461313BC,platform3,10000,10000,2022-05-31 23:57:48,0,0,NaN,0
1357394,b6cf5bf28ae7f52964c6876d4ec9022e,cf0bfb6806a1e98fcae3b0865053d697,748.0,1,NaN,platform1,235900,235900,2022-05-31 23:58:04,0,0,NaN,3
1357395,1a352cadeebce8e075f739b27aa1653b,6466a8119bf99d37314d9f39264a4fbd,748.0,1,NaN,platform1,39206,39206,2022-05-31 23:58:26,0,0,NaN,1
1357396,d683840d76af3e97b29ef09321224dbb,8282c347c6593b49e45b621d46b10647,61.0,1,09F5A3E9-CEED-4E5C-84C7-167E2A3AFAC4,platform1,100000,100000,2022-05-31 23:58:56,0,0,NaN,2


## Giao dịch cùng số tiền

| Mức độ |	Điều kiện |	Giải thích | Điểm |
|:-------|:-----------|:-----------|:-----|
|Cao |	>= 90% giao dịch giống hệt số tiền | Bot lặp script | 3 |
|Trung bình | 70–89% giống nhau | Có thể là gian lận có chủ đích |	2 |
|Thấp |	50–69% | Lặp lại đáng ngờ |	1 |
|Bình thường |	< 50% |	Giao dịch đa dạng |	0 |

In [32]:
# Tính % số giao dịch giống nhau nhất theo từng user
mode_amount_ratio = (
    df_transaction.groupby('userID')['amount']
    .apply(lambda x: x.value_counts(normalize=True).max())
    .reset_index(name='max_amount_ratio')
)

In [34]:
# Hàm tính điểm theo mức giống nhau
def calc_amount_score(ratio):
    if ratio >= 0.9:
        return 3
    elif ratio >= 0.7:
        return 2
    elif ratio >= 0.5:
        return 1
    else:
        return 0

In [35]:
# Tính điểm
mode_amount_ratio['score_amount_repeat'] = mode_amount_ratio['max_amount_ratio'].apply(calc_amount_score)

In [36]:
# Gán điểm lại vào df_transaction
df_transaction = df_transaction.merge(
    mode_amount_ratio[['userID', 'score_amount_repeat']],
    on='userID',
    how='left'
)
df_transaction

,transID,userID,appID,transStatus,deviceID,platform,userChargeAmount,amount,reqDate,campaignID,discountAmount,cashbackTime,score_time_diff,score_amount_repeat
0,2b60a070e297782422257f96ec001e1d,b4cd6bf6334153f048a824fd8c832c06,NaN,1,NaN,NaN,0,50000,2022-03-02 00:00:00,9080,50000,2022-03-02 00:00:00,0,3
1,83f7f779d15c7d64e30197c472e81219,c69b7376c04ace223a597bb495b6433d,NaN,1,NaN,NaN,0,50000,2022-03-02 00:00:00,9080,50000,2022-03-02 00:00:00,0,2
2,c3a55d5a75e4eda4a1c9f1817b6f59f5,2a11b577bfde798dce593467ff6c8472,NaN,1,NaN,NaN,0,50000,2022-03-02 00:00:00,9080,50000,2022-03-02 00:00:00,0,3
3,5660d432965c9081fca164590b86d4e9,898f3699816f0ed166aa5d0c40393cb2,NaN,1,NaN,NaN,0,50000,2022-03-02 00:00:00,9080,50000,2022-03-02 00:00:00,0,3
4,42bddb3b489c9cf2a795e315a0cf4cbe,d1f55da25d8313b0221f513e62e81cc2,15.0,1,8A3E7FFF-87D0-4513-9F5E-D2F6A5DE60FA,platform1,495000,500000,2022-03-02 00:01:01,8901,5000,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1357393,319c7d437de329fdf7e734dae1fcfbc9,b9bab616c0289035af66278ca7c6651a,12.0,1,D189A8C4AB98337A7071E9FD461313BC,platform3,10000,10000,2022-05-31 23:57:48,0,0,NaN,0,1
1357394,b6cf5bf28ae7f52964c6876d4ec9022e,cf0bfb6806a1e98fcae3b0865053d697,748.0,1,NaN,platform1,235900,235900,2022-05-31 23:58:04,0,0,NaN,3,0
1357395,1a352cadeebce8e075f739b27aa1653b,6466a8119bf99d37314d9f39264a4fbd,748.0,1,NaN,platform1,39206,39206,2022-05-31 23:58:26,0,0,NaN,1,0
1357396,d683840d76af3e97b29ef09321224dbb,8282c347c6593b49e45b621d46b10647,61.0,1,09F5A3E9-CEED-4E5C-84C7-167E2A3AFAC4,platform1,100000,100000,2022-05-31 23:58:56,0,0,NaN,2,0


## Spam chiến dịch giới thiệu

| Mức độ |	Điều kiện |	Giải thích | Điểm |
|:-------|:-----------|:-----------|:-----|
|Cao |	≥ 10 lượt mời trong < 1 ngày | Mời ảo liên tục | 8 |
|Trung bình | 5–9 lượt mời trong < 1 ngày | Spam nhẹ |	4 |
|Thấp |	2–4 lượt mời trong < 1 ngày | Có thể là bạn bè thật |	2 |
|Bình thường |	0–1 lượt mời trong < 1 ngày |	Bình thường |	0 |

In [24]:
df = df_referral.copy()
df['reqDate'] = pd.to_datetime(df['reqDate'])
# Trích xuất ngày
df['reqdate'] = df['reqDate'].dt.date

In [25]:
# Tính số người nhận khác nhau mà mỗi user gửi trong cùng 1 ngày
referral_counts = (
    df.groupby(['userID', 'reqDate'])['refereeId']
    .nunique()
    .reset_index(name='unique_referrals')
)

In [26]:
# Tìm max số lượt mời/ngày của mỗi user
max_referrals = (
    referral_counts.groupby('userID')['unique_referrals']
    .max()
    .reset_index(name='max_daily_referrals')
)

In [27]:
# Tính điểm theo điều kiện
def calc_referral_score(n):
    if n >= 10:
        return 8
    elif n >= 5:
        return 4
    elif n >= 2:
        return 2
    else:
        return 0
max_referrals['score_spam_referral'] = max_referrals['max_daily_referrals'].apply(calc_referral_score)

In [28]:
# Gán điểm ngược lại vào df_referral nếu cần
df_referral = df_referral.merge(max_referrals[['userID', 'score_spam_referral']], on='userID', how='left')
df_referral

,userID,refereeId,reqDate,score_spam_referral
0,70ba4ffcc0c22d810444a714ce76549a,a646be9328f3a5e2b2e60356d620dc4f,2022-03-15 11:46:12,0
1,baa20456f723b85f6749489ceaff6c5d,4f82f297f1ed0a9aa499f5bf5bc6b6fb,2022-03-15 11:46:14,0
2,62d1eaadbf019aaeefadea2cbd15d676,38ac6222d194663927310d990047a74d,2022-03-15 11:46:16,0
3,a1a0aefd96fb3c5f4e4ca3d74c215c7f,05de7386213e736a47d04b1874f3fb34,2022-03-15 11:46:18,0
4,c4695bb783ac2a192a014552c3af65a8,9d37504a24d2a13af20e93eac2ff9466,2022-03-15 11:46:36,0
...,...,...,...,...
664938,454436680ed71dc5c82939640b0c4708,e5fbe5db97dc3362772211f8131eca66,2022-05-04 04:38:29,0
664939,be1966b756bd772aa02629c7dd5a4b9f,a362ec513ce9f79d2ae1417ebd892e45,2022-05-04 04:33:49,0
664940,a9270c1ff90a6cdfa0baabfa1d3aeca8,5eb76663b061457a5f2e48365554ab4d,2022-05-04 04:37:16,0
664941,8c204ce0a22a4cdee46aa4bc959a3ce5,ce8c0a3ac6615393f1e7884cdcc9282c,2022-05-04 04:38:52,0


## Một user tham gia nhiều campaign trong thời gian rất ngắn

| Mức độ |	Điều kiện |	Giải thích | Điểm |
|:-------|:-----------|:-----------|:-----|
|Cao |	≥ 5 campaign trong < 10 phút | Tự động farm | 3 |
|Trung bình | 3–4 campaign / 10 phút | Có dấu hiệu |	2 |
|Thấp |	2 campaign / 10 phút | Nghi ngờ | 1 |
|Bình thường |	< 2 campaign / 10 phút |	Bình thường |	0 |

In [55]:
df_tx = df_transaction.copy()

# Chuyển cột thời gian về kiểu datetime
df_tx['reqDate'] = pd.to_datetime(df_tx['reqDate'])

# Giữ các giao dịch thành công
df_success = df_tx[df_tx['transStatus'] == 1]

# Sắp xếp để tính rolling window
df_success = df_success.sort_values(by=['userID', 'reqDate'])

# Xóa những dòng lỗi nếu có
df_success = df_success.dropna(subset=['reqDate'])

In [56]:
# Tạo hàm tính số lượng campaign khác nhau trong sliding window 10 phút
def max_campaigns_in_10min(group):
    times = group['reqDate'].tolist()
    campaigns = group['campaignID'].tolist()
    max_count = 0
    for i in range(len(times)):
        current_time = times[i]
        seen_campaigns = set()
        for j in range(i, len(times)):
            if (times[j] - current_time).total_seconds() <= 600:
                seen_campaigns.add(campaigns[j])
                max_count = max(max_count, len(seen_campaigns))
            else:
                break
    return pd.Series({
        'max_campaigns_10min': max_count
    })


In [57]:
# Áp dụng theo từng user
campaign_window = df_success.groupby('userID').apply(max_campaigns_in_10min).reset_index()


C:\Users\Acer\AppData\Local\Temp\ipykernel_3596\4070423950.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  campaign_window = df_success.groupby('userID').apply(max_campaigns_in_10min).reset_index()


In [58]:
# Gán điểm
def score_campaigns(n):
    if n >= 5:
        return 3
    elif n >= 3:
        return 2
    elif n == 2:
        return 1
    else:
        return 0
campaign_window['score_campaigns'] = campaign_window['max_campaigns_10min'].apply(score_campaigns)

In [59]:
# Gán điểm trở lại df_transaction
df_transaction = df_transaction.merge(campaign_window[['userID', 'score_campaigns']], on='userID', how='left')
df_transaction

,transID,userID,appID,transStatus,deviceID,platform,userChargeAmount,amount,reqDate,campaignID,discountAmount,cashbackTime,score_time_diff,score_amount_repeat,score_campaigns
0,2b60a070e297782422257f96ec001e1d,b4cd6bf6334153f048a824fd8c832c06,NaN,1,NaN,NaN,0,50000,2022-03-02 00:00:00,9080,50000,2022-03-02 00:00:00,0,3,0.0
1,83f7f779d15c7d64e30197c472e81219,c69b7376c04ace223a597bb495b6433d,NaN,1,NaN,NaN,0,50000,2022-03-02 00:00:00,9080,50000,2022-03-02 00:00:00,0,2,1.0
2,c3a55d5a75e4eda4a1c9f1817b6f59f5,2a11b577bfde798dce593467ff6c8472,NaN,1,NaN,NaN,0,50000,2022-03-02 00:00:00,9080,50000,2022-03-02 00:00:00,0,3,0.0
3,5660d432965c9081fca164590b86d4e9,898f3699816f0ed166aa5d0c40393cb2,NaN,1,NaN,NaN,0,50000,2022-03-02 00:00:00,9080,50000,2022-03-02 00:00:00,0,3,0.0
4,42bddb3b489c9cf2a795e315a0cf4cbe,d1f55da25d8313b0221f513e62e81cc2,15.0,1,8A3E7FFF-87D0-4513-9F5E-D2F6A5DE60FA,platform1,495000,500000,2022-03-02 00:01:01,8901,5000,NaN,0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1357393,319c7d437de329fdf7e734dae1fcfbc9,b9bab616c0289035af66278ca7c6651a,12.0,1,D189A8C4AB98337A7071E9FD461313BC,platform3,10000,10000,2022-05-31 23:57:48,0,0,NaN,0,1,1.0
1357394,b6cf5bf28ae7f52964c6876d4ec9022e,cf0bfb6806a1e98fcae3b0865053d697,748.0,1,NaN,platform1,235900,235900,2022-05-31 23:58:04,0,0,NaN,3,0,1.0
1357395,1a352cadeebce8e075f739b27aa1653b,6466a8119bf99d37314d9f39264a4fbd,748.0,1,NaN,platform1,39206,39206,2022-05-31 23:58:26,0,0,NaN,1,0,1.0
1357396,d683840d76af3e97b29ef09321224dbb,8282c347c6593b49e45b621d46b10647,61.0,1,09F5A3E9-CEED-4E5C-84C7-167E2A3AFAC4,platform1,100000,100000,2022-05-31 23:58:56,0,0,NaN,2,0,1.0


## Một user chuyển tiền cho cùng 1 người nhận nhiều lần

| Mức độ |	Điều kiện |	Giải thích | Điểm |
|:-------|:-----------|:-----------|:-----|
|Cao |	≥ 3 lần chuyển cùng người trong cùng 1 ngày | Loop lặp rõ ràng | 3 |
|Trung bình | 2 lần trong cùng 1 ngày | Nghi có chủ đích |	2 |
|Bình thường |	0–1 lần trong cùng 1 ngày |	Giao dịch bình thường |	0 |

In [61]:
df_ts = df_transfer.copy()
# Chuyển thời gian về kiểu datetime và lấy ngày
df_ts['reqDate'] = pd.to_datetime(df_ts['reqDate'])
df_ts['date'] = df_ts['reqDate'].dt.date
# Chỉ giữ giao dịch thành công 
df_ts = df_ts[df_ts['transStatus'] == 1]

In [62]:
# Đếm số lần sender chuyển cho cùng 1 receiver trong 1 ngày
loop_counts = (
    df_ts.groupby(['sender', 'receiver', 'date'])
    .size()
    .reset_index(name='count')
)

In [63]:
# Với mỗi sender, lấy số lần cao nhất chuyển cho cùng 1 người trong 1 ngày
max_loops = (
    loop_counts.groupby('sender')['count']
    .max()
    .reset_index(name='max_loop_same_receiver')
)

In [64]:
# Tính điểm theo điều kiện
def loop_score(n):
    if n >= 3:
        return 3
    elif n == 2:
        return 2
    else:
        return 0

max_loops['score_loop_transfer'] = max_loops['max_loop_same_receiver'].apply(loop_score)


In [66]:
df_transfer = df_transfer.merge(
    max_loops[['sender', 'score_loop_transfer']],
    on='sender',
    how='left'
)

In [68]:
# Nếu user không có giao dịch chuyển tiền thì điền 0
df_transfer['score_loop_transfer'] = df_transfer['score_loop_transfer'].fillna(0).astype(int)
df_transfer

,receiver,sender,transID,transStatus,amount,deviceID,platform,reqDate,score_loop_transfer
0,35ecf363a0db8a7f74d3683a3be081d4,0603beedc2ecf6abc5d712ddc739724f,af95c5d01417e339cbf6aa5358fa264c,1,10000,A34CF1C1996615AB953A3263350F0F23,platform3,2022-03-02 00:00:16,2
1,dcdc146880852d0e88eacd95f611a73d,cce517dedfd140fe44489d7658bead03,b9293d87c906a0a51e94eafadbc96bc5,1,30000,224DD5A1F61DE5547C61C4EADCBEEC35,platform3,2022-03-02 00:01:08,3
2,0889b1ba3ffdcfd2f7c4c1c4cf1a5526,08e46b1044d2df84664537a667f939b8,6a9e330061c77d0280f2f9dd17e6f1a4,1,50000,B26C087A-7D15-4D03-B5BC-BEFD56A951F5,platform1,2022-03-02 00:01:13,3
3,f14767f54aa34d3dbc6cf14e5ba8f215,814d10a5c6a55f033617b6ffd2d43e7d,200b9279737687e1199086557c7e581d,1,250000,03FC3EFB-D6D8-4678-A72E-7CFDAA0E8CA0,platform1,2022-03-02 00:01:18,3
4,0889b1ba3ffdcfd2f7c4c1c4cf1a5526,3b38e89e608b689254bd69c6da3481ac,997e9e4ae78228081f843d4fc26e1361,1,100000,a847161bd5ed4f01,platform2,2022-03-02 00:01:58,2
...,...,...,...,...,...,...,...,...,...
1971433,ab83f0237d476c61eec07b5dbc0af709,b0b70271212479167d533c1180681cd6,1fa7f02091cb852bcc6d964daa39c690,1,15000,D17760632DBFDFEF12C0740172BD7410,platform3,2022-05-31 23:59:42,3
1971434,ab83f0237d476c61eec07b5dbc0af709,fd859ba1b70cd4ac5ac937f7a3ef371a,36e198ebaa7907da72860e1c2197e917,1,50000,F824C9F9FE70E7261B445A97F4F21F83,platform3,2022-05-31 23:59:46,3
1971435,2a2f3b8ea17082bebc5a4c3e2f7ecf5a,d9b671e15b1cc0504a8006b696ff3d1b,5ec470affa2257a48e9280b18767b283,1,10000,32497F57-02EA-47B1-8A67-DC2D96F71933,platform1,2022-05-31 23:59:53,3
1971436,923490eb99cfbddf29354951c8354c25,1e7b07ed90c41319f87fb17726f470bf,eefb4c159836a485b13257f3e04cf5d7,1,100000,6a806fe07f13554e,platform2,2022-05-31 23:59:54,3


## Tính điểm 

In [108]:
result_scoring = pd.read_csv('userID_in_campaign.csv')

In [109]:
result_scoring.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 864342 entries, 0 to 864341
Data columns (total 1 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   userID  864342 non-null  object
dtypes: object(1)
memory usage: 6.6+ MB


In [83]:
df_ts = df_transfer.copy()

In [84]:
df_ts = df_ts.rename(columns={'sender': 'userID'})

In [74]:
df_transaction.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1357398 entries, 0 to 1357397
Data columns (total 15 columns):
 #   Column               Non-Null Count    Dtype  
---  ------               --------------    -----  
 0   transID              1357398 non-null  object 
 1   userID               1357398 non-null  object 
 2   appID                898502 non-null   float64
 3   transStatus          1357398 non-null  int64  
 4   deviceID             734616 non-null   object 
 5   platform             898502 non-null   object 
 6   userChargeAmount     1357398 non-null  int64  
 7   amount               1357398 non-null  int64  
 8   reqDate              1357398 non-null  object 
 9   campaignID           1357398 non-null  int64  
 10  discountAmount       1357398 non-null  int64  
 11  cashbackTime         581461 non-null   object 
 12  score_time_diff      1357398 non-null  int64  
 13  score_amount_repeat  1357398 non-null  int64  
 14  score_campaigns      1354976 non-null  float64
dty

In [110]:
score_cols = ['userID', 'score_time_diff', 'score_amount_repeat', 'score_campaigns']
score_data = df_transaction[score_cols].drop_duplicates(subset='userID')

result_scoring = result_scoring.merge(score_data, on='userID', how='left')

# Fill NaN thành 0
for col in ['score_time_diff', 'score_amount_repeat', 'score_campaigns']:
    result_scoring[col] = result_scoring[col].fillna(0).astype(int)

In [111]:
result_scoring.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 864342 entries, 0 to 864341
Data columns (total 4 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   userID               864342 non-null  object
 1   score_time_diff      864342 non-null  int64 
 2   score_amount_repeat  864342 non-null  int64 
 3   score_campaigns      864342 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 26.4+ MB


In [ ]:
# Lấy cột cần merge và loại trùng userID
score_loop = df_ts[['userID', 'score_loop_transfer']].drop_duplicates(subset='userID')

# Merge vào result_scoring
result_scoring = result_scoring.merge(score_loop, on='userID', how='left')

# Fill NaN = 0 và ép kiểu int
result_scoring['score_loop_transfer'] = result_scoring['score_loop_transfer'].fillna(0).astype(int)
result_scoring.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 864342 entries, 0 to 864341
Data columns (total 5 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   userID               864342 non-null  object
 1   score_time_diff      864342 non-null  int64 
 2   score_amount_repeat  864342 non-null  int64 
 3   score_campaigns      864342 non-null  int64 
 4   score_loop_transfer  864342 non-null  int64 
dtypes: int64(4), object(1)
memory usage: 33.0+ MB


In [113]:
# Lấy cột cần merge và loại trùng userID
score_referral = df_referral[['userID', 'score_spam_referral']].drop_duplicates(subset='userID')

# Merge vào result_scoring
result_scoring = result_scoring.merge(score_referral, on='userID', how='left')

# Fill NaN = 0 và ép kiểu int
result_scoring['score_spam_referral'] = result_scoring['score_spam_referral'].fillna(0).astype(int)
result_scoring.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 864342 entries, 0 to 864341
Data columns (total 6 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   userID               864342 non-null  object
 1   score_time_diff      864342 non-null  int64 
 2   score_amount_repeat  864342 non-null  int64 
 3   score_campaigns      864342 non-null  int64 
 4   score_loop_transfer  864342 non-null  int64 
 5   score_spam_referral  864342 non-null  int64 
dtypes: int64(5), object(1)
memory usage: 39.6+ MB


In [114]:
# Tính tổng điểm từ 5 tiêu chí
score_columns = ['score_time_diff', 'score_amount_repeat', 'score_campaigns', 
                 'score_loop_transfer', 'score_spam_referral']

result_scoring['total_score'] = result_scoring[score_columns].sum(axis=1)


Ta xây dựng một hệ thống chấm điểm gồm 5 tiêu chí, mỗi tiêu chí có mức độ nặng nhẹ (0–3, 0–8 điểm), với mục đích phân biệt người thật và bot dựa trên hành vi. Ta chọn threshold = 5 là một ngưỡng khởi đầu hợp lý vì nó đánh dấu điểm bắt đầu từ người có ≥ 2 tiêu chí rủi ro, tránh overflag người thật, phù hợp với thiết kế hệ thống scoring và phân phối điểm ban đầu. Chọn mức threshold ban đầu bằng 5, sau đó dùng kĩ thuật gini index để thay đổi threshhold cho tối ưu



In [127]:
# Chọn threshold (có thể thay đổi để test)
threshold = 5

# Gán nhãn dự đoán từ total_score
result_scoring['predicted_label'] = (result_scoring['total_score'] >= threshold).astype(int)

In [128]:
result_scoring

,userID,score_time_diff,score_amount_repeat,score_campaigns,score_loop_transfer,score_spam_referral,total_score,predicted_label
0,70ba4ffcc0c22d810444a714ce76549a,0,0,0,0,0,0,0
1,baa20456f723b85f6749489ceaff6c5d,0,0,0,0,0,0,0
2,62d1eaadbf019aaeefadea2cbd15d676,0,0,0,0,0,0,0
3,a1a0aefd96fb3c5f4e4ca3d74c215c7f,0,1,2,0,0,3,0
4,c4695bb783ac2a192a014552c3af65a8,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...
864337,b4d3888f8090313766fa60217e1a297b,0,1,0,0,0,1,0
864338,bf50de5ecd79a858da7dfdb7b08d928f,0,3,0,0,0,3,0
864339,1ce6ec7477f7906b821f0fbdcdce7a76,0,1,0,0,0,1,0
864340,761f1d0dbd1f993bcdcb249c5217a80a,0,1,0,0,0,1,0


## Gini Index

In [134]:
result_original = pd.read_csv('result_original.csv')

In [135]:
# Hàm tính Gini Index
def gini(actual, pred):
    assert len(actual) == len(pred)
    all_data = np.asarray(np.c_[actual, pred, np.arange(len(actual))], dtype=np.float64)
    all_data = all_data[np.lexsort((all_data[:,2], -1 * all_data[:,1]))]
    total_losses = all_data[:,0].sum()
    gini_sum = all_data[:,0].cumsum().sum() / total_losses
    gini_sum -= (len(actual) + 1) / 2.0
    return gini_sum / len(actual)

def gini_normalized(actual, pred):
    return gini(actual, pred) / gini(actual, actual)

# Tính Gini
gini_score = gini_normalized(result_original['is_fraud'], result_scoring['predicted_label'])
print(f"Gini Index: {gini_score:.4f}")

Gini Index: 0.8232


Với Gini = 0.8232, ta thấy mô hình có khả năng phân biệt bot và không bot rất mạnh. Tiếp theo thử nhiều ngưỡng threshold (từ 1 → 15) và xem tại đâu Gini cao nhất.


In [137]:
best_threshold = None
best_gini = -1

for t in range(1, 16):
    result_scoring['predicted_label'] = (result_scoring['total_score'] >= t).astype(int)
    g = gini_normalized(result_original['is_fraud'], result_scoring['predicted_label'])
    if g > best_gini:
        best_gini = g
        best_threshold = t
    print(f"Threshold = {t}, Gini = {g:.4f}")

print(f"\nBest Threshold = {best_threshold}, Gini Index = {best_gini:.4f}")

Threshold = 1, Gini = 0.9295
Threshold = 2, Gini = 0.9254
Threshold = 3, Gini = 0.9119
Threshold = 4, Gini = 0.8385
Threshold = 5, Gini = 0.8232
Threshold = 6, Gini = 0.7954
Threshold = 7, Gini = 0.7630
Threshold = 8, Gini = 0.7585
Threshold = 9, Gini = 0.7486
Threshold = 10, Gini = 0.7428
Threshold = 11, Gini = 0.7388
Threshold = 12, Gini = 0.7381
Threshold = 13, Gini = 0.7372
Threshold = 14, Gini = 0.7372
Threshold = 15, Gini = 0.7371

Best Threshold = 1, Gini Index = 0.9295


Mặc dù Gini Index đạt cao nhất tại threshold = 1 (0.9295), nhưng vì dữ liệu gốc gán nhãn bot rất chặt chẽ, nên hệ thống scoring dễ overflag nếu dùng ngưỡng này. Trong triển khai thực tế, threshold nên đặt từ 5–8 để đảm bảo độ chính xác cao và tránh ảnh hưởng đến user thật. Với threshold từ 5-7 sẽ đánh giá được những user có dấu hiệu bất ngờ, cần kiểm tra thêm, với threshold = 8 sẽ vi phạm nhiều tiêu chí nặng => rất nghi ngờ có thể chặn ngay

Ta tính precision, recall, f1-score, accuracy, false positive rate,... để dễ dàng lựa chọn threshold phù hợp để triển khai thực tế

In [140]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
import pandas as pd

threshold_results = []

for t in range(1, 16):
    result_scoring['predicted_label'] = (result_scoring['total_score'] >= t).astype(int)

    y_true = result_original['is_fraud']
    y_pred = result_scoring['predicted_label']

    # Tính confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    flagged_ratio = y_pred.mean()
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0

    threshold_results.append({
        'threshold': t,
        'precision': round(precision, 4),
        'recall': round(recall, 4),
        'f1_score': round(f1, 4),
        'accuracy': round(acc, 4),
        'flagged_ratio': round(flagged_ratio, 4),
        'false_positives': fp,
        'false_positive_rate': round(false_positive_rate, 4)
    })

# Hiển thị kết quả
df_threshold = pd.DataFrame(threshold_results)
df_threshold


,threshold,precision,recall,f1_score,accuracy,flagged_ratio,false_positives,false_positive_rate
0,1,0.0147,0.9260,0.0289,0.8736,0.1282,109166,0.1266
1,2,0.0174,0.8844,0.0342,0.8984,0.1031,87597,0.1016
2,3,0.0220,0.8047,0.0428,0.9269,0.0744,62864,0.0729
3,4,0.0239,0.5285,0.0458,0.9552,0.0449,37876,0.0439
4,5,0.0274,0.4408,0.0515,0.9670,0.0327,27505,0.0319
5,6,0.0288,0.3069,0.0527,0.9776,0.0216,18160,0.0211
6,7,0.0264,0.1822,0.0462,0.9847,0.0140,11782,0.0137
7,8,0.0574,0.1031,0.0737,0.9947,0.0036,2972,0.0034
8,9,0.0800,0.0547,0.0650,0.9968,0.0014,1104,0.0013
9,10,0.1159,0.0245,0.0404,0.9976,0.0004,328,0.0004


Với mục tiêu ưu tiên, Precision cao => Ai đã bị gán là bot thì gần như chắc chắn là bot. False Positive Rate (FPR) thấp => Ít gắn nhầm người dùng thật. Accept được: recall không quá cao (bỏ sót một vài bot vẫn chấp nhận) 

=> Ta chọn threshold = 9 là ngưỡng tối ưu để cực kỳ hạn chế việc flag nhầm người dùng thật và chấp nhận bỏ sót một phần bot (recall ~5%)

In [143]:
threshold = 9

# Gán nhãn dự đoán từ total_score
result_scoring['predicted_label'] = (result_scoring['total_score'] >= threshold).astype(int)
result_scoring['predicted_label'].value_counts()

predicted_label
0    863142
1      1200
Name: count, dtype: int64

In [144]:
result_scoring.to_csv('result_scoring.csv', index=False)

In [146]:
# Tổng amount từ df_transaction theo userID
amount_tx = df_transaction.groupby('userID', as_index=False)['amount'].sum()
amount_tx = amount_tx.rename(columns={'amount': 'amount_transaction'})

# Tổng amount từ df_transfer theo userID
amount_tf = df_ts.groupby('userID', as_index=False)['amount'].sum()
amount_tf = amount_tf.rename(columns={'amount': 'amount_transfer'})

# Merge lần lượt vào result_scoring
result_scoring = result_scoring.merge(amount_tx, on='userID', how='left')
result_scoring = result_scoring.merge(amount_tf, on='userID', how='left')

# Fill NaN nếu có user không có giao dịch hoặc không chuyển tiền
result_scoring[['amount_transaction', 'amount_transfer']] = result_scoring[
    ['amount_transaction', 'amount_transfer']
].fillna(0)

# Tính tổng amount theo user (tùy chọn)
result_scoring['total_amount'] = result_scoring['amount_transaction'] + result_scoring['amount_transfer']

# Gộp lại với thông tin amount + label nếu cần
df = result_scoring.copy()
df['true_label'] = result_original['is_fraud']

# Tổng amount bị mô hình flag (bất kể đúng sai)
flagged_amount = df[df['predicted_label'] == 1]['total_amount'].sum()

# Tổng amount mô hình flag đúng (True Positive)
tp_amount = df[(df['predicted_label'] == 1) & (df['true_label'] == 1)]['total_amount'].sum()

# Tổng amount của bot thật
total_bot_amount = df[df['true_label'] == 1]['total_amount'].sum()

# Tổng amount bị flag sai (False Positive)
fp_amount = df[(df['predicted_label'] == 1) & (df['true_label'] == 0)]['total_amount'].sum()

# Tổng amount toàn bộ giao dịch
total_amount = df['total_amount'].sum()

# Tính % hiệu quả
print(f"Tổng số tiền toàn bộ: {total_amount:,.0f}")
print(f"Số tiền bị flag: {flagged_amount:,.0f} ({flagged_amount / total_amount:.2%})")
print(f"Số tiền bot thật được ngăn chặn (TP): {tp_amount:,.0f} ({tp_amount / total_bot_amount:.2%} của bot)")
print(f"Số tiền flag sai (FP): {fp_amount:,.0f} ({fp_amount / flagged_amount:.2%} của flagged)")


Tổng số tiền toàn bộ: 263,167,988,369
Số tiền bị flag: 8,017,701,940 (3.05%)
Số tiền bot thật được ngăn chặn (TP): 611,274,225 (9.92% của bot)
Số tiền flag sai (FP): 7,406,427,715 (92.38% của flagged)


Tỷ lệ flag toàn hệ thống thấp (3.05%) => Hệ thống không quá aggressive, không làm phiền phần lớn người dùng. Như ở đề cập ở trên bộ dữ liệu gốc đã đánh giá quá "chặt tay" để xác định bot còn đối với mô hình đánh giá ta chọn threshold để tối ưu triển khai trong thực tế nên dữ liệu thống kê sẽ "xấu".